# Comparing NAEP and PISA: What Can (and Can't) You Ask?

NAEP and PISA are two of the most widely cited math assessment programs, but they
measure different things, in different populations, on different scales. This notebook
walks through:

1. **What each dataset contains** — structure, coverage, and meaning
2. **What's comparable and what isn't** — and *why*
3. **Within-system trend analysis** — the questions each dataset *can* answer
4. **What a cross-system view adds** — and where it misleads

The goal is not to produce one combined chart, but to understand what kinds of
questions are valid to ask of each source — and of both together.

**Runs in Google Colab** — no install required.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

NAEP_URL = "https://raw.githubusercontent.com/rkn2/open-math-insights/main/public/data/naep_math_scale_scores.csv"
PISA_URL = "https://raw.githubusercontent.com/rkn2/open-math-insights/main/public/data/pisa_math_scores.csv"

naep = pd.read_csv(NAEP_URL)
pisa = pd.read_csv(PISA_URL)

print(f"NAEP: {naep.shape[0]} rows, {naep.shape[1]} columns")
print(f"PISA: {pisa.shape[0]} rows, {pisa.shape[1]} columns")

## 1. Side-by-side: what's in each dataset?

Before doing any analysis, let's be precise about what each assessment measures.

In [ ]:
comparison = pd.DataFrame({
    "Feature": [
        "Administered by",
        "Population tested",
        "Sampling frame",
        "Score scale",
        "Scale meaning",
        "Years in this dataset",
        "Frequency",
        "Geographic unit",
        "Content domain",
    ],
    "NAEP": [
        "NCES (U.S. Dept of Education)",
        "U.S. students in grades 4 and 8",
        "Nationally representative + select states",
        "0–500",
        "Linked across years — designed for trend comparison",
        f"{naep['year'].min()}–{naep['year'].max()}",
        "Every 2 years (with gaps)",
        "National + state (6 jurisdictions in this extract)",
        "Mathematics (broad)",
    ],
    "PISA": [
        "OECD",
        "15-year-olds (regardless of grade)",
        "Nationally representative per country",
        "~200–800 (centered at 500 in 2003)",
        "Linked across years — designed for trend comparison",
        f"{pisa['year'].min()}–{pisa['year'].max()}",
        "Every 3 years",
        f"Country-level ({pisa['entity'].nunique()} countries)",
        "Mathematical literacy (applied)",
    ],
})

comparison.style.set_properties(**{"text-align": "left"}).hide(axis="index")

### Key differences that affect analysis

| Difference | Why it matters |
|---|---|
| **Population**: grade 4/8 vs. age 15 | Grade-based and age-based samples overlap only partially. A 15-year-old in the U.S. is typically in grade 9 or 10 — *not* tested by NAEP math. |
| **Scale**: 0–500 vs. ~200–800 | The numbers are not comparable. A NAEP score of 240 and a PISA score of 480 cannot be placed on the same axis. |
| **Content**: curriculum-aligned vs. literacy-focused | NAEP assesses U.S. curriculum standards; PISA assesses applied mathematical literacy. They define "math" differently. |
| **Frequency**: 2-year vs. 3-year cycle | The datasets share some years (2003, 2009, 2015, 2022) but not all. |

## 2. Within-system trends: questions each dataset CAN answer

Both NAEP and PISA are designed for **within-system trend analysis** — comparing a
jurisdiction to its own past performance on a linked scale. These are the strongest
questions each dataset supports.

### NAEP: Has U.S. math performance changed over time?

In [ ]:
national = naep[naep["jurisdiction"] == "NP"].sort_values("year")

fig, ax = plt.subplots(figsize=(10, 5))
for grade, color in [(4, "#2563eb"), (8, "#16a34a")]:
    d = national[national["grade"] == grade]
    ax.plot(d["year"], d["avg_scale_score"], marker="o", color=color,
            linewidth=2, label=f"Grade {grade}")
    # Annotate start and end
    ax.annotate(f"{d['avg_scale_score'].iloc[0]:.0f}",
                (d["year"].iloc[0], d["avg_scale_score"].iloc[0]),
                textcoords="offset points", xytext=(-15, 8), fontsize=9, color=color)
    ax.annotate(f"{d['avg_scale_score'].iloc[-1]:.0f}",
                (d["year"].iloc[-1], d["avg_scale_score"].iloc[-1]),
                textcoords="offset points", xytext=(5, 8), fontsize=9, color=color)

ax.set_title("NAEP National Public Math Scores Over Time", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Avg Scale Score")
ax.legend()
plt.tight_layout()
plt.show()

print("\nValid question: 'Did national math scores recover from the 2022 dip by 2024?'")
print("Valid question: 'Is the grade-4 trend different from grade 8?'")

### NAEP: How do states compare to each other and to the national average?

In [ ]:
grade8 = naep[naep["grade"] == 8].sort_values("year")
nat8 = grade8[grade8["jurisdiction"] == "NP"][["year", "avg_scale_score"]].set_index("year")

fig, ax = plt.subplots(figsize=(12, 5))
states = grade8[grade8["jurisdiction"] != "NP"]

for jur in states["jurisdiction"].unique():
    d = states[states["jurisdiction"] == jur].set_index("year")
    diff = d["avg_scale_score"] - nat8["avg_scale_score"]
    ax.plot(diff.index, diff.values, marker="o", markersize=4,
            label=d["jurisdiction_label"].iloc[0])

ax.axhline(0, color="#94a3b8", linewidth=1, linestyle="--", label="National average")
ax.set_title("Grade 8: State Scores Relative to National Average", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Points above/below national")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\nValid question: 'Is Texas closing the gap with Massachusetts?'")
print("Valid question: 'Did any state's relative position change after the pandemic?'")

### PISA: How does the U.S. compare to other countries over time?

In [ ]:
peers = ["United States", "Canada", "United Kingdom", "Germany", "Japan", "South Korea",
         "Finland", "Australia"]

fig, ax = plt.subplots(figsize=(12, 6))
for country in peers:
    d = pisa[pisa["entity"] == country].sort_values("year")
    if len(d) == 0:
        continue
    style = {"marker": "o", "markersize": 4, "linewidth": 1.5}
    if country == "United States":
        style.update({"linewidth": 3, "color": "#1e3a5f", "zorder": 10})
    ax.plot(d["year"], d["pisa_math_all_average"], label=country, **style)

ax.set_title("PISA Math Scores: U.S. and Peer Countries", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("PISA Math Score")
ax.legend(fontsize=9, loc="lower left")
plt.tight_layout()
plt.show()

print("\nValid question: 'Is the U.S. trend flat while peer countries decline?'")
print("Valid question: 'Which countries maintained scores through the pandemic cycle?'")

## 3. What a cross-system view can (carefully) tell you

You **cannot** compare NAEP and PISA scores directly — the scales are different, the
populations are different, and the content frameworks are different. Putting them on
one axis is misleading.

But you *can* ask: **do the two systems tell the same story about trends?**

If NAEP shows U.S. scores declining and PISA shows U.S. scores stable, that's
interesting — it might mean the decline is concentrated in younger grades (NAEP 4/8)
and hasn't reached 15-year-olds (PISA), or that the two assessments are sensitive
to different aspects of mathematical competence.

Let's check.

In [ ]:
# Normalize both to their own 2003 baseline = 100 (index)
naep_nat = naep[(naep["jurisdiction"] == "NP") & (naep["grade"] == 8)].sort_values("year")
pisa_us = pisa[pisa["entity"] == "United States"].sort_values("year")

naep_base = naep_nat[naep_nat["year"] == 2003]["avg_scale_score"].values[0]
pisa_base = pisa_us[pisa_us["year"] == 2003]["pisa_math_all_average"].values[0]

naep_indexed = naep_nat[["year", "avg_scale_score"]].copy()
naep_indexed["index"] = (naep_indexed["avg_scale_score"] / naep_base) * 100

pisa_indexed = pisa_us[["year", "pisa_math_all_average"]].copy()
pisa_indexed["index"] = (pisa_indexed["pisa_math_all_average"] / pisa_base) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(naep_indexed["year"], naep_indexed["index"], marker="o",
        color="#2563eb", linewidth=2, label="NAEP Grade 8 (National)")
ax.plot(pisa_indexed["year"], pisa_indexed["index"], marker="s",
        color="#dc2626", linewidth=2, label="PISA (U.S.)")
ax.axhline(100, color="#94a3b8", linewidth=1, linestyle="--", label="2003 baseline")
ax.set_title("U.S. Math Performance: Indexed to 2003 = 100", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Index (2003 = 100)")
ax.legend()
plt.tight_layout()
plt.show()

print("This chart normalizes each series to its own 2003 value, so the Y axis")
print("shows relative change, not absolute level. That's the only valid way to")
print("put these two series on one plot.")

### Interpreting the indexed comparison

Both series hover near 100 for most of the period — the U.S. shows no dramatic improvement
in either system. The 2022 dip is visible in both, suggesting the pandemic effect was real
and showed up regardless of which assessment you look at.

**What this does NOT tell you:**
- Whether U.S. students are "good" or "bad" at math in absolute terms
- Whether NAEP and PISA measure the same thing
- Whether changes in one system *caused* changes in the other

**What it DOES tell you:**
- The *direction* of U.S. trends is roughly consistent across two independent assessments
- The pandemic-era dip is not an artifact of one assessment's design

## 4. Questions you can ask — and questions you can't

Use this as a quick reference when deciding what to do with these datasets.

In [ ]:
questions = pd.DataFrame({
    "Question": [
        "Has U.S. grade 4 math improved since 2003?",
        "Has U.S. grade 8 math improved since 2003?",
        "How does Texas compare to Massachusetts?",
        "How does the U.S. compare to Finland?",
        "Did the pandemic affect math scores?",
        "Is the U.S. 'above average' in math?",
        "Are NAEP scores higher than PISA scores?",
        "Which countries improved the most since 2003?",
        "Do both systems agree on the U.S. trend direction?",
        "Can I compare a NAEP state to a PISA country?",
    ],
    "Dataset": [
        "NAEP", "NAEP", "NAEP", "PISA", "Both",
        "PISA", "Neither", "PISA", "Both", "Neither",
    ],
    "Valid?": [
        "Yes", "Yes", "Yes", "Yes (trend, not level)",
        "Yes (each independently)",
        "Careful — relative to PISA OECD average, with caveats",
        "No — different scales, different populations",
        "Yes (within-PISA trend)",
        "Yes — index to a common baseline year",
        "No — different populations, scales, and content",
    ],
})

questions.style.set_properties(**{"text-align": "left"}).hide(axis="index")

## 5. Exercise: try your own questions

Here are some prompts to explore on your own — modify the code cells above or
write new ones:

1. **NAEP**: Which state had the biggest score change between 2019 and 2024?
   Does the pattern differ between grade 4 and grade 8?

2. **PISA**: Pick three countries on different continents. How do their trajectories
   compare? Is there a global trend or is each trajectory idiosyncratic?

3. **Both**: Index NAEP grade 4 (instead of grade 8) against PISA. Does the trend
   agreement change? What would that mean?

4. **Critical thinking**: A news article claims "U.S. math scores are the lowest
   they've been in decades." Using these datasets, how would you evaluate that
   claim? What additional data would you want?

In [ ]:
# Your code here


## Further reading

- [NAEP Data Explorer](https://www.nationsreportcard.gov/ndecore/landing) — query the
  full NAEP microdata
- [PISA Data Explorer](https://pisadataexplorer.oecd.org/) — query full PISA results
  including subscales and student questionnaires
- [OMI Data Depot](https://rkn2.github.io/open-math-insights/data-depot) —
  browse and download the datasets used in this notebook
- [OMI Researcher Guide](https://rkn2.github.io/open-math-insights/researcher-guide) —
  how to de-identify and share your own education data